In [0]:
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark.conf.set("spark.sql.execution.arrow.pyspark.maxRecordsPerBatch", 1)
#spark.conf.set("spark.task.resource.memory", "4g")
#spark.conf.set("spark.sql.files.maxPartitionBytes", 1 * 1024 * 1024)
#spark.conf.set("spark.task.resource.myresource.amount", "1")
#spark.conf.set("spark.executor.resource.myresource.amount", "1")
#spark.conf.get("spark.speculation")

In [0]:
dbutils.widgets.text("input_folder", "Hackathon4/", "Input DICOM Folder")
dbutils.widgets.text("input_parquet", "Hackathon4_output_v4_large_files.parquet", "Input parquet for large files")
dbutils.widgets.text("output_parquet", "Hackathon4_large_files_output", "Output Parquet")
dbutils.widgets.text("file_size_threshold", "5242880", "File size threshold (bytes)")
dbutils.widgets.text("batch_size", "1000", "Batch Size")
dbutils.widgets.text("sampling_size", "0", "Sampling Size (0=disabled)")


In [0]:
for widget in ["input_folder", "input_parquet", "output_parquet", "file_size_threshold", "batch_size", "sampling_size"]:
    print(f"{widget}: {dbutils.widgets.get(widget)}")

In [0]:
import pydicom
from tqdm import tqdm
#import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import os
from io import BytesIO
from pydicom.filebase import DicomFileLike
from openai import OpenAI
from PIL import Image
import base64



In [0]:
input_dir = f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get("input_folder")}"

In [0]:

df = spark.read.parquet(f"/Volumes/1_inland/sectra/vone/{dbutils.widgets.get("input_parquet")}")

In [0]:
display(df.limit(1000))

In [0]:
import time
import pandas as pd
from pathlib import Path
from pydicom.encaps import get_frame, generate_pixel_data_frame
import gc

api_key = dbutils.secrets.get(scope = "adc_store", key = "image-anon_gpt41mini_api-key")


def extract_dicom_tags(file_path):
    ds_header = pydicom.dcmread(file_path, stop_before_pixels=True)
    accession_number = ds_header.get("AccessionNumber", None)
    modality = ds_header.get("Modality", None)
    return accession_number, modality


def extract_first_frame(file_path):
    """Extract frame 0 from any DICOM. Memory-safe for multi-frame files."""
    # Read header to check frame count
    ds_header = pydicom.dcmread(file_path, stop_before_pixels=True)
    rows = ds_header.Rows
    cols = ds_header.Columns
    bits = ds_header.BitsAllocated
    num_frames = int(ds_header.NumberOfFrames) if "NumberOfFrames" in ds_header else 1
    samples = ds_header.SamplesPerPixel if "SamplesPerPixel" in ds_header else 1

    del ds_header

    # Estimate full uncompressed size
    full_size_mb = rows * cols * (bits // 8) * num_frames * samples / (1024**2)

    ds = pydicom.dcmread(file_path, stop_before_pixels=False)

    if num_frames > 1 and full_size_mb > 50:
        # Large multi-frame: extract only first compressed frame
        is_encapsulated = (
            hasattr(ds["PixelData"], "is_undefined_length")
            and ds["PixelData"].is_undefined_length
        )
        if is_encapsulated:
            try:
                frame_bytes = get_frame(ds.PixelData, 0)
            except Exception:
                frame_bytes = next(generate_pixel_data_frame(ds.PixelData, num_frames))
            del ds
            gc.collect()
            return np.array(Image.open(BytesIO(frame_bytes)))
        else:
            # Uncompressed multi-frame: slice raw bytes for frame 0
            frame_nbytes = rows * cols * (bits // 8) * samples
            raw = ds.PixelData[:frame_nbytes]
            del ds
            dtype = np.uint8 if bits == 8 else np.uint16
            return np.frombuffer(raw, dtype=dtype).reshape(rows, cols)
    else:
        # Single frame or small multi-frame: safe to decompress all
        arr = ds.pixel_array
        if arr.ndim == 3 and num_frames > 1:
            frame = arr[0].copy()
            del arr
        else:
            frame = arr
        del ds
        return frame
    
def frame_to_b64(frame_arr):
    """Resize frame to 512x512 and encode as base64 PNG."""
    img = Image.fromarray(frame_arr)
    del frame_arr
    img = img.resize((512, 512), Image.LANCZOS)

    arr_small = np.array(img, dtype=np.float32)
    del img
    arr_min, arr_max = arr_small.min(), arr_small.max()
    if arr_max > arr_min:
        arr_small = ((arr_small - arr_min) * (255.0 / (arr_max - arr_min))).astype(np.uint8)
    else:
        arr_small = np.zeros(arr_small.shape, dtype=np.uint8)

    img_out = Image.fromarray(arr_small)
    del arr_small
    buf = BytesIO()
    img_out.save(buf, format="PNG", optimize=True)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    buf.close()
    del img_out
    return b64

def llm_pii_detection(row):
    system_prompt = "Output the result in a structured format with 4 required fields: is_medical_scan, is_text_detected, is_personal_information_detected (e.g. name, ID numbers), and confidence_score. Valid values for is_medical_scan are yes, no, and not_sure. Answer yes only if the input image is a typical medical scan of an anatomical structure. Answer no if the input image is a screen capture, graph, or document etc. Valid values for is_text_detected are yes, no, and not_sure. Valid values for is_personal_information_detected are yes, no, and not_sure. confidence_score is an integer from 0 to 100. 0 indicates high confidence of no text detected. 100 indicates high confidence of text being detected. 50 indicates uncertain. Do not leave any field empty. Always respond with 4 fields everytime."

    user_prompt = "Tell me if there is any text in the image. Also tell me if these texts contain any personal information e.g. name, date of birth, scan date, ID numbers. Then give me a confidence score."

    deployment_name = "gpt-4.1-mini"

    endpoint = "https://image-anonymisation-resource.cognitiveservices.azure.com/openai/v1/"



    client = OpenAI(base_url=f"{endpoint}",api_key=api_key)


    #for path, content in zip(pdf['path'], pdf['content']):
    path = row['path']
    try:
        start_time_str = time.strftime("%Y-%m-%d %H:%M:%S")
        is_loaded_ok = False
        accession_number = None
        modality = None

        #ds = pydicom.dcmread(path[5:], stop_before_pixels=False)

        # Get pixel data (no extra copy yet)
        #arr = ds.pixel_array  # already numpy
        path = path[5:]
        frame = extract_first_frame(path)
        base64_str = frame_to_b64(frame)
        del frame
        result = None
        accession_number = None
        is_loaded_ok = True
        modality = None




        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text", "text": user_prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{base64_str}"}
                        }

                    ]
                }
            ]
        )

        result = completion.choices[0].message.content
        accession_number, modality = extract_dicom_tags(path)
        
        return dict(path=path, is_loaded_ok=is_loaded_ok, llm_result=result,  accession_number=accession_number, modality=modality, error_message=None, start_ts=start_time_str, end_ts=time.strftime("%Y-%m-%d %H:%M:%S"))

    except Exception as e:
        return dict(path=path, is_loaded_ok=is_loaded_ok, llm_result=result,  accession_number=accession_number, modality=modality, error_message=str(e)[:1000], start_ts=start_time_str, end_ts=time.strftime("%Y-%m-%d %H:%M:%S"))


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType, BooleanType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("llm_result", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("modality", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("start_ts", StringType(), True),
    StructField("end_ts", StringType(), True)
])



In [0]:
results = []
for row in tqdm(df.collect()):
    # Access row fields by name, e.g. row['path']
    results.append(llm_pii_detection(row))

results_df = spark.createDataFrame(results, schema=output_schema)
display(results_df)
